In [1]:
import pairadigm as pdm
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from typing import List, Tuple, Optional, Union, Dict
import numpy as np
from tqdm import tqdm
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import pearsonr
from pairadigm import Pairadigm
import copy
import random
import polars as pl

import sys
from datetime import datetime
import os

/fs/clip-projects/clip-airreading/mlchrzan/miniconda/envs/emopair/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set random seeds for reproducibility
def set_seeds(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Detect best device available
def get_best_device(cuda_index: int = 0, verbose: bool = True):
    """Detect best device: CUDA (with index) > MPS > CPU"""
    if torch.cuda.is_available():
        if cuda_index < torch.cuda.device_count():
            device = torch.device(f"cuda:{cuda_index}")
            device_name = f"cuda:{cuda_index}"
            if verbose:
                try:
                    print(f"Using NVIDIA GPU {cuda_index}: {torch.cuda.get_device_name(cuda_index)}")
                except Exception:
                    print(f"Using NVIDIA GPU {cuda_index}")
        else:
            raise ValueError(f"CUDA device {cuda_index} not available. Available devices: 0 to {torch.cuda.device_count()-1}")

    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        device_name = "mps"
        if verbose:
            print("Using Apple Silicon GPU via MPS")
    else:
        device = torch.device("cpu")
        device_name = "cpu"
        if verbose:
            print("No GPU found, using CPU")
    return device, device_name

# Set seeds and get device
seed_num = 42
set_seeds(seed=seed_num)
device, device_name = get_best_device(cuda_index=0, verbose=True)

Using NVIDIA GPU 0: NVIDIA RTX 4000 Ada Generation


In [3]:
# Load emopair dataset (contains id, split, V, text, Aro/Dom BT scores)
print("Loading data...")
emopair = pl.read_csv("../Results/emopair.csv").to_pandas()
emopair_pairs = pl.read_csv("../Results/emopair_pairs_w_breakdowns.csv").to_pandas()

print(f"Data loaded. {len(emopair)} EmoPair items. {len(emopair_pairs)} pairs.")

Loading data...
Data loaded. 10062 EmoPair items. 103836 pairs.


In [9]:
emopair

,item_id,split,V,A,D,text,A_BT_score_full,D_BT_score_full,A_rescaled,D_rescaled,A_BT_score_split,D_BT_score_split
0,110CYL068_1036_1079,train,3.00,3.00,3.20,"Remember what she said in my last letter? """,-0.007953,-0.249618,0.000,0.100,-0.080316,-0.311499
1,110CYL068_1079_1110,test,2.80,3.10,2.80,If I wasn't working here.,-0.165819,-0.574112,0.050,-0.100,-0.218244,-0.621851
2,110CYL068_1127_1130,train,3.00,3.00,3.00,"..""",-0.286353,-0.330882,0.000,0.000,-0.655125,-0.606478
3,110CYL068_1137_1188,train,3.44,3.00,3.22,Goodwill helps people get off of public assist...,-0.146604,0.134371,0.000,0.110,-0.251934,0.226989
4,110CYL068_1189_1328,train,3.55,3.27,3.46,Sherry learned through our Future Works class ...,0.170138,0.214993,0.135,0.230,0.320377,0.421976
...,...,...,...,...,...,...,...,...,...,...,...,...
10057,wwf12_4531_4624,train,3.00,3.50,3.00,Please let it be a constant reminder of all yo...,0.268549,-0.128251,0.250,0.000,0.605149,-0.409790
10058,wwf12_501_591,train,3.80,3.40,3.60,That’s why I want to extend my appreciation fo...,-0.067112,0.133948,0.200,0.300,-0.251579,0.129240
10059,wwf12_592_691,train,3.00,3.00,3.10,And why I’m writing you today to ask you to re...,0.150444,-0.047152,0.000,0.050,0.206740,-0.360066
10060,wwf12_702_921,train,3.33,3.44,3.44,"In fact, I want to urge you to strengthen your...",0.366853,0.214345,0.220,0.220,0.775513,0.324047


In [7]:
# Split data into train/dev/test based on 'split' column in emopair_pairs
train_set_pairs = emopair_pairs[(emopair_pairs['item1_split'] == 'train') & (emopair_pairs['item2_split'] == 'train')].copy()
print(f"Training set prepared. Number of pairs: {len(train_set_pairs)}")

eval_set_pairs = emopair_pairs[(emopair_pairs['item1_split'] == 'dev') & (emopair_pairs['item2_split'] == 'dev')].copy()
print(f"Evaluation set prepared. Number of pairs: {len(eval_set_pairs)}")

test_set_pairs = emopair_pairs[(emopair_pairs['item1_split'] == 'test') & (emopair_pairs['item2_split'] == 'test')].copy()
print(f"Test set prepared. Number of pairs: {len(test_set_pairs)}")

Training set prepared. Number of pairs: 46664
Evaluation set prepared. Number of pairs: 8451
Test set prepared. Number of pairs: 8485


In [8]:
emopair_pairs

,pair_id,item1,item2,item1_split,item2_split,item1_text,item2_text,item1_V,item2_V,item1_A,...,item1_D,item2_D,A_breakdown1,A_breakdown2,A_decision,A_justification,D_breakdown1,D_breakdown2,D_decision,D_justification
0,110CYL070_264_371___SemEval_1116,110CYL070_264_371,SemEval_1116,train,train,We've expanded Goodwill's proven methods to to...,"Planned cesareans not risk-free, group warns",3.40,2.90,3.10,...,3.10,2.90,Original Text: We've expanded Goodwill's prove...,Original Text: Planned cesareans not risk-free...,Text2,FINAL ANSWER: Description 2 \nJUSTIFICATION: ...,Original Text: We've expanded Goodwill's prove...,Original Text: Planned cesareans not risk-free...,Text1,FINAL ANSWER: Description 1\n\nJUSTIFICATION: ...
1,SemEval_653___Uprooted_Farming-on-Sand_3146_3237,SemEval_653,Uprooted_Farming-on-Sand_3146_3237,train,train,The energy challenge | Exotic visions: how to ...,The Provincetown farmers market has lovely pro...,3.10,3.00,3.10,...,3.00,3.00,Original Text: The energy challenge | Exotic v...,Original Text: The Provincetown farmers market...,Text1,FINAL ANSWER: Description 1 \nJUSTIFICATION: ...,Original Text: The energy challenge | Exotic v...,Original Text: The Provincetown farmers market...,Text2,FINAL ANSWER: Description 2\n\nJUSTIFICATION: ...
2,SemEval_619___WhatToHongKong_4104_4137,SemEval_619,WhatToHongKong_4104_4137,train,train,Sleep more to fight childhood obesity,Hong Kong is full of giant malls.,3.00,3.37,3.09,...,3.18,3.27,Original Text: Sleep more to fight childhood o...,Original Text: Hong Kong is full of giant mall...,Text1,FINAL ANSWER: Description 1 \nJUSTIFICATION: ...,Original Text: Sleep more to fight childhood o...,Original Text: Hong Kong is full of giant mall...,Text1,FINAL ANSWER: Description 1\n\nJUSTIFICATION: ...
3,115CVL036_1074_1315___easy_money_9883_9898,115CVL036_1074_1315,easy_money_9883_9898,train,train,"With this in mind, I am asking each one of you...",I was in shock.,3.30,2.20,3.30,...,3.40,2.80,"Original Text: With this in mind, I am asking ...",Original Text: I was in shock.\nPrompt 1 respo...,Text2,FINAL ANSWER: Description 2 \nJUSTIFICATION: ...,"Original Text: With this in mind, I am asking ...",Original Text: I was in shock.\nPrompt 1 respo...,Text1,FINAL ANSWER: Description 1\n\nJUSTIFICATION: ...
4,Madame_White_Snake_1782_2043___captured_moment...,Madame_White_Snake_1782_2043,captured_moments_3713_3806,train,train,"“She was a loving wife, a caring mother, rescu...","Were they mood-sensitive, they would have chan...",3.46,3.10,2.82,...,3.00,3.20,"Original Text: “She was a loving wife, a carin...","Original Text: Were they mood-sensitive, they ...",Text1,FINAL ANSWER: Description 1 \nJUSTIFICATION: ...,"Original Text: “She was a loving wife, a carin...","Original Text: Were they mood-sensitive, they ...",Text2,FINAL ANSWER: Description 2\n\nJUSTIFICATION: ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103831,hotel-california_28048_28089___The_Black_Willo...,hotel-california_28048_28089,The_Black_Willow_20833_21069,test,dev,I cornered Adrienne at dinner that night.,He considered Arthur's intervention; considere...,2.90,3.30,3.10,...,3.60,3.20,Original Text: I cornered Adrienne at dinner t...,Original Text: He considered Arthur's interven...,Text1,FINAL ANSWER: Description 1 \nJUSTIFICATION: ...,Original Text: I cornered Adrienne at dinner t...,Original Text: He considered Arthur's interven...,Text1,FINAL ANSWER: Description 1\n\nJUSTIFICATION: ...
103832,wsj_0124_839_932___detroit_6926_6963,wsj_0124_839_932,detroit_6926_6963,test,dev,"In addition, operating results were hit by an ...",(west of Livernois around McNichols).,2.80,3.00,2.90,...,3.10,3.00,"Original Text: In addition, operating results ...",Original Text: (west of Livernois around McNic...,Text1,FINAL ANSWER: Description 1 \nJUSTIFICATION: ...,"Original Text: In addition, operating results ...",Original Text: (west of Livernois around McNic...,Text1,FINAL ANSWER: Desc